# MAS v2 Baseline — 3DSRBench

Main architecture (no Trust Score): Head Agent → 3 Specialists → Final Reasoning Agent.

**Benchmark:** 3DSRBench (ccvl/3DSRBench)  
**Sample sizes:** 10, 50, 100  
**Train/Test split:** 50/50

In [ ]:
import sys
from pathlib import Path

# Project root
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "run_eval_mas_v2.py").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from run_eval_mas_v2 import build_runners, run_experiment
from src2.agents.mas_v2 import ScoreMapUpdater

In [ ]:
# H100: use local DeepSeek-R1 (no API server)
head_gen, spec_gen, reason_gen = build_runners(
    specialist_device="cuda",
    use_local_reasoning=True,
    reasoning_local_model_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
)

In [ ]:
BENCHMARK = "3dsrbench"
SAMPLE_SIZES = [10, 50, 100]
SEED = 42
OUTPUT_BASE = ROOT / "results" / "mas_v2_baseline" / BENCHMARK

In [ ]:
results_summary = []

for n in SAMPLE_SIZES:
    print(f"\n{'='*60}")
    print(f"3DSRBench | {n} samples")
    print("="*60)
    out_dir = str(OUTPUT_BASE / f"{n}samples")
    out = run_experiment(
        benchmark=BENCHMARK,
        head_generate=head_gen,
        specialist_generate=spec_gen,
        reasoning_generate=reason_gen,
        train_ratio=0.5,
        seed=SEED,
        output_dir=out_dir,
        max_samples=n,
    )
    results_summary.append({
        "samples": n,
        "train_acc": out["train_metrics"]["accuracy"],
        "test_acc": out["test_metrics"]["accuracy"],
        "train_n": out["train_metrics"]["total"],
        "test_n": out["test_metrics"]["total"],
    })

In [ ]:
import pandas as pd

df = pd.DataFrame(results_summary)
df["train_acc%"] = (df["train_acc"] * 100).round(1)
df["test_acc%"] = (df["test_acc"] * 100).round(1)
display(df[["samples", "train_n", "test_n", "train_acc%", "test_acc%"]])